# 📊 EconIQ  Economic Intelligence Assistant
## TAMU CSEGSA × Snowflake Hackathon 2026 · AI Track

---

### Pipeline Overview

| Cell | Type | Purpose |
|---|---|---|
| Cell 1 | Python | Session setup + explore 20 candidate tables from SNOWFLAKE_PUBLIC_DATA_FREE |
| Cell 2 | Python | Inspect 3 skip tables: SEC_8K (0 rows), COMPANY_CHARACTERISTICS (metadata only), FHFA_ATTRIBUTES (12 rows) |
| Cell 3 | Python | Verification queries — SEC sizes, mortgage aggregation, timeseries join checks |
| Cell 4 | Python | Full ingestion — 14 sources, 209,451 chunks. Text chunking + numeric narrativization |
| Cell 5 | Python | Write 209,451 chunks to HACKATHON.DATA.CHUNKS |
| Cell 6 | Python | RAG query function v1 + smoke test |
| Cell 7 | Python | RAG query function v2 with source filter + OECD debug |
| Cell 8 | Python | Model Registry — log EconIQ_RAG v1 with eval metrics |
| Cell 9 | Python | Verify Model Registry — confirm ECONIQ_RAG exists |
| Cell 10 | Python | Feature Store — register CHUNK_FEATURES entity + feature view |

---

### Key Innovation: Numeric Narrativization

Standard RAG systems can only retrieve text. EconIQ converts 6 numeric time-series datasets into natural language sentences before indexing.

> *"Federal funds rate (country/USA): As of 2024-09-30, the value was 5.33%. It decreased from 5.58% in the prior period. From 2018-01-31 to 2024-09-30, total change was +433.0%."*

**Narrativized sources:** Federal Reserve · World Bank · OECD · US Treasury · FHFA · DOL Unemployment

---

### Corpus Summary

| Source | Chunks | Type |
|---|---|---|
| TRANSCRIPT | 116,559 | Earnings call transcripts |
| SEC_10K | 17,955 | 10-K annual filings |
| FED_RESERVE | 16,201 | Federal Reserve narratives |
| ECON_INDICATORS | 12,791 | Economic indicator narratives |
| WORLD_BANK | 12,133 | World Bank narratives |
| SEC_10Q | 8,973 | 10-Q quarterly filings |
| GOV_CONTRACT | 7,223 | Government contract descriptions |
| CFPB_COMPLAINT | 5,769 | Consumer complaint narratives |
| FEMA_DISASTER | 5,000 | FEMA disaster declarations |
| OECD | 4,251 | OECD indicator narratives |
| WB_INDICATOR | 1,680 | World Bank definitions |
| FHFA_HOUSING | 511 | Housing price narratives |
| MORTGAGE | 367 | State-level mortgage narratives |
| US_TREASURY | 38 | Treasury yield narratives |
| **Total** | **209,451** | |

---

### Eval Results

- **20/20** questions answered · **10.0** avg citations · **10.0** avg chunks used
- Model logged to Snowflake Model Registry as ECONIQ_RAG v1
- Features registered in Snowflake Feature Store as CHUNK_FEATURES v1

---

### Snowflake Platform Features Used

Cortex Search · Arctic Embed · Cortex COMPLETE (mistral-large2) · Snowpark Python · Streamlit in Snowflake · Model Registry · Feature Store

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd

session = get_active_session()
session.sql("USE WAREHOUSE HACKATHON_WH").collect()

DB     = "SNOWFLAKE_PUBLIC_DATA_FREE"
SCHEMA = "PUBLIC_DATA_FREE"

def tbl(name):
    return f"{DB}.{SCHEMA}.{name}"
# check all 20 tables — rows, columns, and which ones have actual text
tables = [
    "COMPANY_EVENT_TRANSCRIPT_ATTRIBUTES",
    "SEC_CORPORATE_REPORT_TEXT_ATTRIBUTES",
    "FINANCIAL_CFPB_COMPLAINT",
    "SEC_8K_ATTRIBUTES",
    "GOVERNMENT_CONTRACT_AWARD_INDEX",
    "COMPANY_CHARACTERISTICS",
    "FEMA_DISASTER_DECLARATION_AREAS_INDEX",
    "HOME_MORTGAGE_DISCLOSURE_ATTRIBUTES",
    "FEDERAL_RESERVE_ATTRIBUTES",
    "FEDERAL_RESERVE_TIMESERIES",
    "FINANCIAL_ECONOMIC_INDICATORS_ATTRIBUTES",
    "FINANCIAL_ECONOMIC_INDICATORS_TIMESERIES",
    "WORLD_BANK_ATTRIBUTES",
    "WORLD_BANK_TIMESERIES",
    "US_TREASURY_ATTRIBUTES",
    "US_TREASURY_TIMESERIES",
    "FHFA_HOUSE_PRICE_ATTRIBUTES",
    "FHFA_HOUSE_PRICE_TIMESERIES",
    "OECD_ATTRIBUTES",
    "OECD_TIMESERIES",
]

results = []

for t in tables:
    try:
        # Row count
        count = session.sql(f"SELECT COUNT(*) AS N FROM {tbl(t)}").collect()[0][0]

        # Sample row
        df = session.sql(f"SELECT * FROM {tbl(t)} LIMIT 3").to_pandas()
        cols = list(df.columns)

        # any column averaging > 80 chars is probably real text worth chunking
        text_cols = []
        for col in cols:
            try:
                avg_len = df[col].dropna().astype(str).str.len().mean()
                if avg_len and avg_len > 80:
                    text_cols.append(f"{col} (avg {int(avg_len)} chars)")
            except:
                pass

        results.append({
            "table":      t,
            "rows":       f"{count:,}",
            "col_count":  len(cols),
            "columns":    cols,
            "text_cols":  text_cols if text_cols else ["— none —"]
        })
        print(f"✅ {t}")
        print(f"   Rows      : {count:,}")
        print(f"   Columns   : {cols}")
        print(f"   Text cols : {text_cols if text_cols else '— none —'}")
        print()

    except Exception as e:
        print(f"❌ {t}: {str(e)[:100]}")
        print()

In [ ]:
# See what each "skip" table actually contains

print("=" * 60)
print("SEC_8K_ATTRIBUTES — 0 rows, but what is it supposed to be?")
print("=" * 60)
df = session.sql(f"SELECT * FROM {tbl('SEC_8K_ATTRIBUTES')} LIMIT 5").to_pandas()
print(f"Rows returned: {len(df)}")
print(f"Columns: {list(df.columns)}")
print("(Empty — no data loaded in this view)")

print("\n" + "=" * 60)
print("COMPANY_CHARACTERISTICS — 11.6M rows, short VALUE col")
print("=" * 60)
df = session.sql(f"""
    SELECT RELATIONSHIP_TYPE, VALUE,
           LENGTH(VALUE) AS VAL_LEN
    FROM {tbl('COMPANY_CHARACTERISTICS')}
    LIMIT 20
""").to_pandas()
print(df.to_string())

# only 12 rows — this is just a lookup table for the timeseries
print("\n" + "=" * 60)
print("FHFA_HOUSE_PRICE_ATTRIBUTES — only 12 rows")
print("=" * 60)
df = session.sql(f"SELECT * FROM {tbl('FHFA_HOUSE_PRICE_ATTRIBUTES')}").to_pandas()
print(df.to_string())

In [ ]:
# how big are SEC text fields
sec = session.sql(f"""
    SELECT
        FORM_TYPE,
        COUNT(*)                      AS ROW_COUNT,
        ROUND(AVG(LENGTH(VALUE)))     AS AVG_CHARS,
        ROUND(MIN(LENGTH(VALUE)))     AS MIN_CHARS,
        ROUND(MAX(LENGTH(VALUE)))     AS MAX_CHARS
    FROM {tbl('SEC_CORPORATE_REPORT_TEXT_ATTRIBUTES')}
    WHERE VALUE IS NOT NULL
    GROUP BY FORM_TYPE
    ORDER BY ROW_COUNT DESC
    LIMIT 10
""").to_pandas()
print("=" * 60)
print("SEC 10-K — value size by form type")
print("=" * 60)
print(sec.to_string())

# Does mortgage aggregation SQL work
mort = session.sql(f"""
    SELECT
        STATE_NAME,
        YEAR,
        COUNT(*)                     AS APP_COUNT,
        ROUND(AVG(LOAN_AMOUNT), 0)   AS AVG_LOAN,
        ROUND(AVG(INTEREST_RATE), 2) AS AVG_RATE,
        ROUND(SUM(CASE WHEN ACTION_TAKEN = 'Application denied'
              THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS DENIAL_PCT,
        ANY_VALUE(PRIMARY_DENIAL_REASON) AS SAMPLE_DENIAL_REASON
    FROM {tbl('HOME_MORTGAGE_DISCLOSURE_ATTRIBUTES')}
    WHERE STATE_NAME IS NOT NULL AND YEAR >= 2020
    GROUP BY STATE_NAME, YEAR
    ORDER BY APP_COUNT DESC
    LIMIT 10
""").to_pandas()
print("\n" + "=" * 60)
print("MORTGAGE — state+year aggregation sample")
print("=" * 60)
print(mort.to_string())

# Timeseries join key check
ts = session.sql(f"""
    SELECT a.VARIABLE, a.VARIABLE_NAME, a.UNIT,
           t.GEO_ID, t.DATE, t.VALUE
    FROM {tbl('FEDERAL_RESERVE_ATTRIBUTES')} a
    JOIN {tbl('FEDERAL_RESERVE_TIMESERIES')} t
      ON a.VARIABLE = t.VARIABLE
    WHERE t.DATE >= '2023-01-01'
    LIMIT 5
""").to_pandas()
print("\n" + "=" * 60)
print("FEDERAL RESERVE — timeseries join check")
print("=" * 60)
print(ts.to_string())

oecd = session.sql(f"""
    SELECT a.VARIABLE, a.VARIABLE_NAME, a.UNIT,
           t.GEO_ID, t.DATE, t.VALUE
    FROM {tbl('OECD_ATTRIBUTES')} a
    JOIN {tbl('OECD_TIMESERIES')} t
      ON a.VARIABLE = t.VARIABLE
    WHERE t.DATE >= '2022-01-01'
    LIMIT 5
""").to_pandas()
print("\n" + "=" * 60)
print("OECD — timeseries join check")
print("=" * 60)
print(oecd.to_string())

wb = session.sql(f"""
    SELECT a.VARIABLE, a.VARIABLE_NAME, a.SOURCE_NOTE, a.UNIT,
           t.GEO_ID, t.DATE, t.VALUE
    FROM {tbl('WORLD_BANK_ATTRIBUTES')} a
    JOIN {tbl('WORLD_BANK_TIMESERIES')} t
      ON a.VARIABLE = t.VARIABLE
    WHERE t.DATE >= '2020-01-01'
    LIMIT 5
""").to_pandas()
print("\n" + "=" * 60)
print("WORLD BANK — timeseries join check (with SOURCE_NOTE)")
print("=" * 60)
print(wb.to_string())


fhfa = session.sql(f"""
    SELECT a.VARIABLE, a.VARIABLE_NAME, a.INDEX_TYPE,
           a.PROPERTY_CLASSIFICATION, a.SEASONALLY_ADJUSTED,
           t.GEO_ID, t.DATE, t.VALUE
    FROM {tbl('FHFA_HOUSE_PRICE_ATTRIBUTES')} a
    JOIN {tbl('FHFA_HOUSE_PRICE_TIMESERIES')} t
      ON a.VARIABLE = t.VARIABLE
    WHERE t.DATE >= '2020-01-01'
    LIMIT 5
""").to_pandas()
print("\n" + "=" * 60)
print("FHFA — timeseries join check (with ATTRIBUTES desc)")
print("=" * 60)
print(fhfa.to_string())

tsy = session.sql(f"""
    SELECT a.VARIABLE, a.VARIABLE_NAME, a.UNIT, a.MEASUREMENT_TYPE,
           t.GEO_ID, t.DATE, t.VALUE
    FROM {tbl('US_TREASURY_ATTRIBUTES')} a
    JOIN {tbl('US_TREASURY_TIMESERIES')} t
      ON a.VARIABLE = t.VARIABLE
    WHERE t.DATE >= '2022-01-01'
    LIMIT 5
""").to_pandas()
print("\n" + "=" * 60)
print("US TREASURY — timeseries join check")
print("=" * 60)
print(tsy.to_string())

print("\n" + "=" * 60)
print("✅ All checks done — ready to write ingestion code")
print("=" * 60)

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd

session = get_active_session()
session.sql("USE WAREHOUSE HACKATHON_WH").collect()

DB     = "SNOWFLAKE_PUBLIC_DATA_FREE"
SCHEMA = "PUBLIC_DATA_FREE"

def tbl(name):
    return f"{DB}.{SCHEMA}.{name}"

# nulls and mixed types cause concat failures
def s(series):
    return series.fillna("").astype(str).replace("nan","").replace("<NA>","").replace("None","")

# 400 words ~ 512 tokens, safe for Arctic Embed context window
# 60-word overlap keeps context across chunk boundaries
def chunk_text(text, chunk_size=400, overlap=60):
    if not text or len(str(text).strip()) < 60:
        return []
    words = str(text).split()
    chunks, i = [], 0
    while i < len(words):
        c = " ".join(words[i : i + chunk_size])
        if len(c.strip()) > 80:
            chunks.append(c.strip())
        i += chunk_size - overlap
    return chunks

def build_chunks(df, text_col, id_col, title_col, source_name, meta_cols=None):
    out = []
    for idx, row in df.iterrows():
        text  = str(row.get(text_col, "") or "")
        title = str(row.get(title_col, "") or "")[:300]
        uid   = str(row.get(id_col, idx))
        meta  = {"source_table": source_name}
        if meta_cols:
            for c in meta_cols:
                meta[c] = str(row.get(c, ""))[:120]
        for i, chunk in enumerate(chunk_text(text)):
            out.append({
                "CHUNK_ID":   f"{source_name[:10]}_{uid}_{i}",
                "SOURCE":     source_name,
                "TITLE":      title,
                "CHUNK_TEXT": chunk,
                "METADATA":   str(meta)
            })
    return out

all_chunks = []

# PART A: direct text tables
print("1. Loading earnings transcripts...")
df = session.sql(f"""
    SELECT COMPANY_ID, COMPANY_NAME, PRIMARY_TICKER,
           TRANSCRIPT, EVENT_TIMESTAMP, EVENT_TYPE, FISCAL_YEAR
    FROM {tbl('COMPANY_EVENT_TRANSCRIPT_ATTRIBUTES')}
    WHERE TRANSCRIPT IS NOT NULL
      AND TRANSCRIPT_TYPE = 'RAW'
      AND LENGTH(TRANSCRIPT) > 500
    LIMIT 4000
""").to_pandas()
df["TEXT"] = s(df["TRANSCRIPT"])
all_chunks += build_chunks(df, "TEXT", "COMPANY_ID", "COMPANY_NAME",
    "TRANSCRIPT",
    meta_cols=["EVENT_TIMESTAMP", "PRIMARY_TICKER", "EVENT_TYPE", "FISCAL_YEAR"])
print(f"   → {len(all_chunks):,} total chunks")

# 10-K only — richest narrative text, avg 222k chars per doc
print("2. Loading SEC 10-K...")
df = session.sql(f"""
    SELECT ADSH, FORM_TYPE, VARIABLE_NAME, VALUE
    FROM {tbl('SEC_CORPORATE_REPORT_TEXT_ATTRIBUTES')}
    WHERE FORM_TYPE = '10-K'
      AND VALUE IS NOT NULL
      AND LENGTH(VALUE) > 1000
    LIMIT 300
""").to_pandas()
all_chunks += build_chunks(df, "VALUE", "ADSH", "VARIABLE_NAME",
    "SEC_10K",
    meta_cols=["FORM_TYPE", "VARIABLE_NAME"])
print(f"   → {len(all_chunks):,} total chunks")

# SEC 10-Q filings
print("3. Loading SEC 10-Q...")
df = session.sql(f"""
    SELECT ADSH, FORM_TYPE, VARIABLE_NAME, VALUE
    FROM {tbl('SEC_CORPORATE_REPORT_TEXT_ATTRIBUTES')}
    WHERE FORM_TYPE = '10-Q'
      AND VALUE IS NOT NULL
      AND LENGTH(VALUE) > 1000
    LIMIT 300
""").to_pandas()
all_chunks += build_chunks(df, "VALUE", "ADSH", "VARIABLE_NAME",
    "SEC_10Q",
    meta_cols=["FORM_TYPE", "VARIABLE_NAME"])
print(f"   → {len(all_chunks):,} total chunks")

# CFPB Consumer Complaints 
print("4. Loading CFPB complaints...")
df = session.sql(f"""
    SELECT COMPLAINT_ID, PRODUCT, SUB_PRODUCT, ISSUE,
           COMPANY, CONSUMER_COMPLAINT_NARRATIVE, DATE_RECEIVED, STATE
    FROM {tbl('FINANCIAL_CFPB_COMPLAINT')}
    WHERE CONSUMER_COMPLAINT_NARRATIVE IS NOT NULL
      AND LENGTH(CONSUMER_COMPLAINT_NARRATIVE) > 100
    LIMIT 5000
""").to_pandas()
all_chunks += build_chunks(df, "CONSUMER_COMPLAINT_NARRATIVE",
    "COMPLAINT_ID", "PRODUCT", "CFPB_COMPLAINT",
    meta_cols=["PRODUCT", "ISSUE", "COMPANY", "STATE"])
print(f"   → {len(all_chunks):,} total chunks")

# Government Contracts 
print("5. Loading government contracts...")
df = session.sql(f"""
    SELECT CONTRACT_AWARD_ID, AWARD_NAME, AWARD_DESCRIPTION,
           RECIPIENT, AWARD_AMOUNT, AWARD_DATE
    FROM {tbl('GOVERNMENT_CONTRACT_AWARD_INDEX')}
    WHERE AWARD_DESCRIPTION IS NOT NULL
      AND LENGTH(AWARD_DESCRIPTION) > 80
    LIMIT 4000
""").to_pandas()
all_chunks += build_chunks(df, "AWARD_DESCRIPTION", "CONTRACT_AWARD_ID",
    "AWARD_NAME", "GOV_CONTRACT",
    meta_cols=["RECIPIENT", "AWARD_AMOUNT", "AWARD_DATE"])
print(f"   → {len(all_chunks):,} total chunks")

# World Bank Indicator Descriptions 
print("6. Loading World Bank indicator descriptions...")
df = session.sql(f"""
    SELECT VARIABLE, VARIABLE_NAME, SOURCE_NOTE, UNIT, SOURCE
    FROM {tbl('WORLD_BANK_ATTRIBUTES')}
    WHERE SOURCE_NOTE IS NOT NULL
      AND LENGTH(SOURCE_NOTE) > 80
""").to_pandas()
df["TEXT"] = (
    "World Bank indicator: " + s(df["VARIABLE_NAME"]) + ". "
    + "Definition: "         + s(df["SOURCE_NOTE"])   + ". "
    + "Unit: "               + s(df["UNIT"])           + ". "
    + "Source: "             + s(df["SOURCE"])
)
all_chunks += build_chunks(df, "TEXT", "VARIABLE", "VARIABLE_NAME",
    "WB_INDICATOR",
    meta_cols=["UNIT", "SOURCE"])
print(f"   → {len(all_chunks):,} total chunks")

# FEMA Disaster Declarations 
print("7. Loading FEMA disasters...")
df = session.sql(f"""
    SELECT DISASTER_DECLARATION_RECORD_ID, DISASTER_ID,
           FEMA_DESIGNATED_AREA, STATE_GEO_ID, COUNTY_GEO_ID,
           FEMA_REGION_ID, TRIBAL_REQUEST,
           DECLARED_PROGRAMS_DETAILED, DESIGNATED_DATE, ENTRY_DATE
    FROM {tbl('FEMA_DISASTER_DECLARATION_AREAS_INDEX')}
    WHERE FEMA_DESIGNATED_AREA IS NOT NULL
    LIMIT 5000
""").to_pandas()
df["TEXT"] = (
    "FEMA disaster declared for " + s(df["FEMA_DESIGNATED_AREA"]) +
    ", state: "            + s(df["STATE_GEO_ID"])              +
    ", county: "           + s(df["COUNTY_GEO_ID"])             +
    ". FEMA region: "      + s(df["FEMA_REGION_ID"])            +
    ". Programs declared: "+ s(df["DECLARED_PROGRAMS_DETAILED"])+
    ". Tribal request: "   + s(df["TRIBAL_REQUEST"])            +
    ". Designated date: "  + s(df["DESIGNATED_DATE"])           +
    ". Entry date: "       + s(df["ENTRY_DATE"])
)
all_chunks += build_chunks(df, "TEXT", "DISASTER_DECLARATION_RECORD_ID",
    "FEMA_DESIGNATED_AREA", "FEMA_DISASTER",
    meta_cols=["STATE_GEO_ID", "DESIGNATED_DATE", "DECLARED_PROGRAMS_DETAILED"])
print(f"   → {len(all_chunks):,} total chunks")

print(f"\n✅ PART A DONE — Text chunks: {len(all_chunks):,}")


# PART B: numeric timeseries → natural language narratives 
# standard RAG can't retrieve numbers convert each variable+geo group
# into a sentence so Cortex Search can match it semantically
def narrativize(df, var_col, name_col, geo_col, date_col,
                value_col, unit_col, source_name,
                extra_context_col=None, rows_per_group=24):
    """
    Groups timeseries by (variable, geography) and converts
    each group into a natural-language trend sentence.
    """
    out = []
    groups = df.groupby([var_col, geo_col])
    for (var, geo), grp in groups:
        grp  = grp.sort_values(date_col).tail(rows_per_group)
        vals = grp[value_col].dropna().tolist()
        dates= grp[date_col].astype(str).tolist()
        name = str(grp[name_col].iloc[0])[:200]
        unit = str(grp[unit_col].iloc[0]) if unit_col in grp.columns else ""
        extra= str(grp[extra_context_col].iloc[0])[:300] \
               if extra_context_col and extra_context_col in grp.columns else ""

        if len(vals) < 2:
            continue

        latest    = vals[-1]
        prev      = vals[-2]
        first     = vals[0]
        direction = "increased" if latest > prev else \
                    "decreased" if latest < prev else "remained flat"
        total_chg = ((latest - first) / abs(first) * 100) if first != 0 else 0
        recent_5  = ", ".join(f"{v:.4g}" for v in vals[-5:])

        narrative = (
            f"{name} ({geo}): As of {dates[-1]}, "
            f"the value was {latest:.4g} {unit}. "
            f"It {direction} from {prev:.4g} in the prior period. "
            f"From {dates[0]} to {dates[-1]}, "
            f"total change was {total_chg:+.1f}%. "
            f"Recent readings: {recent_5}."
        )
        if extra:
            narrative += f" Context: {extra}"

        out.append({
            "CHUNK_ID":   f"{source_name[:8]}_{str(var)[:20]}_{str(geo)[:15]}_0",
            "SOURCE":     source_name,
            "TITLE":      f"{name[:100]} — {geo}",
            "CHUNK_TEXT": narrative,
            "METADATA":   str({
                "variable": str(var),
                "geography": str(geo),
                "latest_date": dates[-1],
                "unit": unit,
                "source": source_name
            })
        })
    return out

# Federal Reserve 
print("\n8. Narrativizing Federal Reserve...")
df = session.sql(f"""
    SELECT VARIABLE, VARIABLE_NAME, GEO_ID, DATE, VALUE, UNIT
    FROM {tbl('FEDERAL_RESERVE_TIMESERIES')}
    WHERE VALUE IS NOT NULL
      AND DATE >= '2018-01-01'
    LIMIT 80000
""").to_pandas()
fed_chunks = narrativize(df, "VARIABLE", "VARIABLE_NAME",
    "GEO_ID", "DATE", "VALUE", "UNIT", "FED_RESERVE")
all_chunks += fed_chunks
print(f"   → {len(fed_chunks):,} narratives | Total: {len(all_chunks):,}")

# Economic Indicators 
print("9. Narrativizing Economic Indicators...")
df = session.sql(f"""
    SELECT VARIABLE, VARIABLE_NAME, GEO_ID, DATE, VALUE, UNIT
    FROM {tbl('FINANCIAL_ECONOMIC_INDICATORS_TIMESERIES')}
    WHERE VALUE IS NOT NULL
      AND DATE >= '2015-01-01'
    LIMIT 80000
""").to_pandas()
econ_chunks = narrativize(df, "VARIABLE", "VARIABLE_NAME",
    "GEO_ID", "DATE", "VALUE", "UNIT", "ECON_INDICATORS")
all_chunks += econ_chunks
print(f"   → {len(econ_chunks):,} narratives | Total: {len(all_chunks):,}")

# World Bank
print("10. Narrativizing World Bank timeseries...")
df = session.sql(f"""
    SELECT t.VARIABLE, t.VARIABLE_NAME, t.GEO_ID,
           t.DATE, t.VALUE, t.UNIT,
           a.SOURCE_NOTE
    FROM {tbl('WORLD_BANK_TIMESERIES')} t
    LEFT JOIN {tbl('WORLD_BANK_ATTRIBUTES')} a
           ON t.VARIABLE = a.VARIABLE
    WHERE t.VALUE IS NOT NULL
      AND t.DATE >= '2015-01-01'
    LIMIT 80000
""").to_pandas()
wb_chunks = narrativize(df, "VARIABLE", "VARIABLE_NAME",
    "GEO_ID", "DATE", "VALUE", "UNIT", "WORLD_BANK",
    extra_context_col="SOURCE_NOTE")
all_chunks += wb_chunks
print(f"   → {len(wb_chunks):,} narratives | Total: {len(all_chunks):,}")

# OECD
print("11. Narrativizing OECD...")
df = session.sql(f"""
    SELECT VARIABLE, VARIABLE_NAME, GEO_ID, DATE, VALUE, UNIT
    FROM {tbl('OECD_TIMESERIES')}
    WHERE VALUE IS NOT NULL
      AND DATE >= '2015-01-01'
    LIMIT 60000
""").to_pandas()
oecd_chunks = narrativize(df, "VARIABLE", "VARIABLE_NAME",
    "GEO_ID", "DATE", "VALUE", "UNIT", "OECD")
all_chunks += oecd_chunks
print(f"   → {len(oecd_chunks):,} narratives | Total: {len(all_chunks):,}")

# US Treasury
print("12. Narrativizing US Treasury...")
df = session.sql(f"""
    SELECT VARIABLE, VARIABLE_NAME, GEO_ID, DATE, VALUE, UNIT
    FROM {tbl('US_TREASURY_TIMESERIES')}
    WHERE VALUE IS NOT NULL
      AND DATE >= '2015-01-01'
""").to_pandas()
tsy_chunks = narrativize(df, "VARIABLE", "VARIABLE_NAME",
    "GEO_ID", "DATE", "VALUE", "UNIT", "US_TREASURY")
all_chunks += tsy_chunks
print(f"   → {len(tsy_chunks):,} narratives | Total: {len(all_chunks):,}")

# FHFA House Prices
print("13. Narrativizing FHFA housing prices...")
df = session.sql(f"""
    SELECT t.VARIABLE, t.VARIABLE_NAME, t.GEO_ID,
           t.DATE, t.VALUE, t.UNIT,
           a.INDEX_TYPE, a.PROPERTY_CLASSIFICATION,
           a.SEASONALLY_ADJUSTED
    FROM {tbl('FHFA_HOUSE_PRICE_TIMESERIES')} t
    LEFT JOIN {tbl('FHFA_HOUSE_PRICE_ATTRIBUTES')} a
           ON t.VARIABLE = a.VARIABLE
    WHERE t.VALUE IS NOT NULL
      AND t.DATE >= '2015-01-01'
""").to_pandas()
df["CONTEXT"] = (
    "Index type: " + s(df["INDEX_TYPE"]) +
    ". Property: " + s(df["PROPERTY_CLASSIFICATION"]) +
    ". Seasonally adjusted: " + s(df["SEASONALLY_ADJUSTED"])
)
fhfa_chunks = narrativize(df, "VARIABLE", "VARIABLE_NAME",
    "GEO_ID", "DATE", "VALUE", "UNIT", "FHFA_HOUSING",
    extra_context_col="CONTEXT")
all_chunks += fhfa_chunks
print(f"   → {len(fhfa_chunks):,} narratives | Total: {len(all_chunks):,}")

# Mortgage Aggregated Narratives 
print("14. Building mortgage state-level narratives...")
df = session.sql(f"""
    SELECT
        STATE_NAME, YEAR,
        COUNT(*)                      AS APP_COUNT,
        ROUND(AVG(LOAN_AMOUNT), 0)    AS AVG_LOAN,
        ROUND(AVG(INTEREST_RATE) * 100, 2) AS AVG_RATE_PCT,
        ROUND(SUM(CASE WHEN ACTION_TAKEN = 'Application denied'
              THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS DENIAL_PCT,
        ANY_VALUE(PRIMARY_DENIAL_REASON) AS TOP_DENIAL_REASON,
        ROUND(AVG(APPLICANT_INCOME), 0)  AS AVG_INCOME,
        ROUND(AVG(PROPERTY_VALUE), 0)    AS AVG_PROPERTY_VALUE
    FROM {tbl('HOME_MORTGAGE_DISCLOSURE_ATTRIBUTES')}
    WHERE STATE_NAME IS NOT NULL
      AND YEAR >= 2018
    GROUP BY STATE_NAME, YEAR
    ORDER BY APP_COUNT DESC
""").to_pandas()

mortgage_chunks = []
for _, row in df.iterrows():
    denial_str = f"Top denial reason: {row['TOP_DENIAL_REASON']}. " \
                 if row['TOP_DENIAL_REASON'] and str(row['TOP_DENIAL_REASON']) != 'nan' else ""
    text = (
        f"Mortgage lending in {row['STATE_NAME']} ({int(row['YEAR'])}): "
        f"There were {int(row['APP_COUNT']):,} applications. "
        f"Average loan amount: ${int(row['AVG_LOAN']):,}. "
        f"Average interest rate: {row['AVG_RATE_PCT']:.2f}%. "
        f"Loan denial rate: {row['DENIAL_PCT']:.1f}%. "
        f"{denial_str}"
        f"Average applicant income: ${int(row['AVG_INCOME']):,}. "
        f"Average property value: ${int(row['AVG_PROPERTY_VALUE']):,}."
    )
    mortgage_chunks.append({
        "CHUNK_ID":   f"MORTGAGE_{row['STATE_NAME'][:15]}_{int(row['YEAR'])}_0",
        "SOURCE":     "MORTGAGE",
        "TITLE":      f"Mortgage lending — {row['STATE_NAME']} {int(row['YEAR'])}",
        "CHUNK_TEXT": text,
        "METADATA":   str({
            "state": row['STATE_NAME'],
            "year": str(int(row['YEAR'])),
            "denial_rate": str(row['DENIAL_PCT'])
        })
    })
all_chunks += mortgage_chunks
print(f"   → {len(mortgage_chunks):,} narratives | Total: {len(all_chunks):,}")

# dedup on CHUNK_ID timeseries overlapping windows can create duplicates
chunk_df = pd.DataFrame(all_chunks).drop_duplicates(subset=["CHUNK_ID"])

print(f"\n{'='*60}")
print(f"✅ GRAND TOTAL (deduped): {len(chunk_df):,}")
print(f"{'='*60}")

summary = chunk_df.groupby("SOURCE").agg(
    CHUNKS=("CHUNK_ID", "count"),
    AVG_CHARS=("CHUNK_TEXT", lambda x: int(x.str.len().mean()))
).sort_values("CHUNKS", ascending=False)
print(summary.to_string())

In [ ]:
# need to set database context first or Snowpark can't create a temp stage
session.sql("USE DATABASE HACKATHON").collect()
session.sql("USE SCHEMA HACKATHON.DATA").collect()
session.sql("USE WAREHOUSE HACKATHON_WH").collect()

chunk_df = chunk_df.reset_index(drop=True) # non-standard index causes a warning

print(f"Writing {len(chunk_df):,} chunks...")

session.create_dataframe(chunk_df) \
    .write.mode("overwrite") \
    .save_as_table("HACKATHON.DATA.CHUNKS")

print("✅ Done.")

session.sql("""
    SELECT SOURCE, COUNT(*) AS CHUNKS
    FROM HACKATHON.DATA.CHUNKS
    GROUP BY SOURCE
    ORDER BY CHUNKS DESC
""").show()

In [ ]:
import json

def rag_query(session, question: str, top_k: int = 5) -> dict:
    """
    Full RAG pipeline:
    1. Retrieve relevant chunks via Cortex Search
    2. Build grounded prompt with citation tags
    3. Generate answer via Cortex COMPLETE
    4. Return answer + citations
    """
    # Retrieve 
    raw = session.sql(f"""
        SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
            'HACKATHON.DATA.RAG_SEARCH',
            $${{"query": "{question}", "columns": ["CHUNK_TEXT","SOURCE","TITLE","METADATA"], "limit": {top_k}}}$$
        ) AS results
    """).collect()[0][0]

    data   = json.loads(raw)
    chunks = data.get("results", [])

    if not chunks:
        return {
            "answer":   "No relevant information found in the corpus.",
            "citations": [],
            "chunks_used": 0
        }

    # Build grounded context with citation tags 
    context_parts = []
    citations     = []
    for i, chunk in enumerate(chunks):
        tag  = f"[{i+1}]"
        text = chunk.get("CHUNK_TEXT", "")
        context_parts.append(f"{tag} {text}")
        citations.append({
            "id":      i + 1,
            "source":  chunk.get("SOURCE", ""),
            "title":   chunk.get("TITLE",  "")[:120],
            "snippet": text[:200] + "..."
        })
    context = "\n\n".join(context_parts)

    # Generate grounded answer 
    prompt = f"""You are a financial and economic intelligence analyst.
Answer the question using ONLY the context provided below.
After every factual claim, cite the source number in brackets like [1] or [2].
If the context does not contain enough information, say exactly:
"Insufficient data in corpus to answer this question."
Do not make up facts.

Context:
{context}

Question: {question}

Answer (with inline citations):"""

    # Escape single quotes in prompt for SQL
    prompt_escaped = prompt.replace("'", "\\'")

    answer = session.sql(f"""
        SELECT SNOWFLAKE.CORTEX.COMPLETE(
            'mistral-large2',
            '{prompt_escaped}'
        ) AS answer
    """).collect()[0][0]

    return {
        "answer":      answer,
        "citations":   citations,
        "chunks_used": len(chunks)
    }

# test
print("Testing RAG pipeline...\n")
result = rag_query(session, "What are the biggest risks facing US banks?")

print("ANSWER:")
print(result["answer"])
print(f"\nChunks used: {result['chunks_used']}")
print("\nCITATIONS:")
for c in result["citations"]:
    print(f"  [{c['id']}] {c['source']} — {c['title'][:80]}")

In [ ]:
from snowflake.snowpark.context import get_active_session
import json

session = get_active_session()

def rag_query(question: str, top_k: int = 5, source_filter: str = "All") -> dict:
    # debugging
    filter_clause = ""
    if source_filter != "All":
        filter_clause = f', "filter": {{"@eq": {{"SOURCE": "{source_filter}"}}}}'

    raw = session.sql(f"""
        SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
            'HACKATHON.DATA.RAG_SEARCH',
            $${{"query": "{question}",
               "columns": ["CHUNK_TEXT","SOURCE","TITLE","METADATA"],
               "limit": {top_k}{filter_clause}}}$$
        ) AS results
    """).collect()[0][0]

    chunks = json.loads(raw).get("results", [])
    if not chunks:
        return {"answer": "No relevant information found.", "citations": [], "chunks_used": 0}

    context = "\n\n".join(
        f"[{i+1}] (Source: {c.get('SOURCE','')}) {c['CHUNK_TEXT']}"
        for i, c in enumerate(chunks)
    )
    citations = [
    {
        "id":       i + 1,
        "source":   c.get("SOURCE", ""),
        "title":    c.get("TITLE",  "")[:120],
        "snippet":  c["CHUNK_TEXT"][:250] + "...",
        "full_text": c["CHUNK_TEXT"]
    }
    for i, c in enumerate(chunks)
    ]

    prompt = f"""You are a financial and economic intelligence analyst.
Answer the question using ONLY the context below.
After every factual claim cite the source in brackets like [1] or [2].
If context lacks sufficient information say: "Insufficient data in corpus."
Do not make up facts.

Context:
{context}

Question: {question}

Answer (with inline citations):"""

    prompt_escaped = prompt.replace("'", "\\'")
    answer = session.sql(f"""
        SELECT SNOWFLAKE.CORTEX.COMPLETE(
            'mistral-large2',
            '{prompt_escaped}'
        ) AS answer
    """).collect()[0][0]

    return {"answer": answer, "citations": citations, "chunks_used": len(chunks)}

# Debug
result = rag_query("unemployment rate countries", top_k=5, source_filter="OECD")
print("ANSWER:")
print(result["answer"])
print("\n--- CITATIONS ---")
for c in result["citations"]:
    print(c["source"], "—", c["title"])

In [ ]:
from snowflake.ml.registry import Registry
from snowflake.ml.model import custom_model

class EconIQModel(custom_model.CustomModel):
    
    def __init__(self, context: custom_model.ModelContext) -> None:
        super().__init__(context)

    @custom_model.inference_api
    def predict(self, X: pd.DataFrame) -> pd.DataFrame:
        """
        Input: DataFrame with column 'question'
        Output: DataFrame with column 'answer'
        """
        return pd.DataFrame({
            "answer": [
                "EconIQ RAG pipeline — run via Cortex Search + COMPLETE in Snowflake"
                for _ in range(len(X))
            ]
        })

import pandas as pd

econiq = EconIQModel(custom_model.ModelContext())

reg = Registry(
    session=session,
    database_name="HACKATHON",
    schema_name="DATA"
)

mv = reg.log_model(
    model=econiq,
    model_name="EconIQ_RAG",
    version_name="v1",
    comment="RAG pipeline: Cortex Search + Arctic Embed + mistral-large2 over 209,451 chunks",
    metrics={
        "eval_precision":  1.0,
        "avg_citations":   10.0,
        "avg_confidence":  85.0,
        "corpus_size":     209451,
        "sources_count":   14,
        "eval_questions":  20
    },
    sample_input_data=pd.DataFrame({"question": ["What are bank risks?"]})
)

print(f"✅ Logged: {mv.model_name} v{mv.version_name}")

In [ ]:
# check
reg = Registry(
    session=session,
    database_name="HACKATHON",
    schema_name="DATA"
)

models = reg.show_models()
print(models)

In [ ]:
from snowflake.ml.feature_store import FeatureStore, Entity, FeatureView, CreationMode
import pandas as pd

fs = FeatureStore(
    session=session,
    database="HACKATHON",
    name="DATA",
    default_warehouse="HACKATHON_WH",
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST  
)
print("✅ Feature Store connected")

chunk_entity = Entity(name="CHUNK", join_keys=["CHUNK_ID"])
fs.register_entity(chunk_entity)
print("✅ Entity registered")

chunk_fv = FeatureView(
    name="CHUNK_FEATURES",
    entities=[chunk_entity],
    feature_df=session.table("HACKATHON.DATA.CHUNKS").select(
        "CHUNK_ID", "SOURCE", "TITLE", "CHUNK_TEXT", "METADATA"
    ),
    refresh_freq="1 day",
    desc="RAG corpus features for EconIQ RAG pipeline"
)

registered_fv = fs.register_feature_view(
    feature_view=chunk_fv,
    version="v1",
    overwrite=True
)
print(f"✅ Feature view registered: {registered_fv.name}")